In [2]:
import pandas as pd

In [3]:
hotel_data_path = '../data/raw/hotel_booking.csv'

In [4]:
raw_df = pd.read_csv(hotel_data_path)

In [5]:
raw_df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date', 'name', 'email',
       'phone-number', 'credit_card'],
      dtype='object')

In [6]:
raw_df.shape[0]*.3

35817.0

# Review data set

In [7]:
raw_df['children'].value_counts()

children
0.0     110796
1.0       4861
2.0       3652
3.0         76
10.0         1
Name: count, dtype: int64

In [8]:
raw_df[['babies', 'is_canceled']].groupby('babies').mean()

,is_canceled
babies,
0,0.371874
1,0.183333
2,0.133333
9,0.000000
10,0.000000


In [9]:
raw_df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

# Remove specific Columns

## missing values
threshold = 30% missing = 35,817</br>
stays_in_weekend_nights 0 = 51,998</br>
children: 0 = 110,796</br>
babies: 0 = 118,473</br>
is_repeated_gust: 0 = 115,580</br>
previous_cancellations: 0 = 112,906</br>
previous_bookings_not_cancelled: 0 = 115,770<br>
booking_changes: 0 = 101,314</br>
days_in_waiting_list: 0 = 115,692</br>
required_car_parking_spaces: 0 = 111,974</br>
total_of_special_requests: 0 = 70,318</br>



## attributes to remove + reason
reservation_status: already captured, encoded in is_canceled where No-Show is also encoded as a cancelation.

reservation_status_date: already encoded in columns arrival_date_year, arrival_date_month, arrival_date_week_number and arrival_date_day_of_the_month

name, email, phonen number, and credit card: do not need these attributes for a regression model.

company and agent: Company has a large number of missing values; agent and company are not relevant to the analysis and can be dropped. 

market_segment and distribution_channel - duplicative information; choose one

In [10]:
log_data = raw_df.copy()

In [11]:
log_data = raw_df.drop(columns=['name', 'email',
       'phone-number', 'credit_card','agent','company','reservation_status','reservation_status_date'])

In [61]:
log_data.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 28 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   is_canceled                     119390 non-null  int64  
 1   lead_time                       119390 non-null  int64  
 2   arrival_date_year               119390 non-null  int64  
 3   arrival_date_week_number        119390 non-null  int64  
 4   arrival_date_day_of_month       119390 non-null  int64  
 5   stays_in_weekend_nights         119390 non-null  int64  
 6   stays_in_week_nights            119390 non-null  int64  
 7   adults                          119390 non-null  int64  
 8   is_repeated_guest               119390 non-null  int64  
 9   previous_cancellations          119390 non-null  int64  
 10  previous_bookings_not_canceled  119390 non-null  int64  
 11  booking_changes                 119390 non-null  int64  
 12  days_in_waiting_

In [62]:
log_data[log_data.columns[19:]].info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 9 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   room_type_mismatch              119390 non-null  int64
 1   reserved_room_type_encoded      119390 non-null  int64
 2   assigned_room_type_encoded      119390 non-null  int64
 3   customer_type_encoded           119390 non-null  int64
 4   arrival_date_month_encode       119390 non-null  int64
 5   distribution_channel_Direct     119390 non-null  bool 
 6   distribution_channel_GDS        119390 non-null  bool 
 7   distribution_channel_TA/TO      119390 non-null  bool 
 8   distribution_channel_Undefined  119390 non-null  bool 
dtypes: bool(4), int64(5)
memory usage: 5.0 MB


In [14]:
log_data[['market_segment']].value_counts()

market_segment
Online TA         56477
Offline TA/TO     24219
Groups            19811
Direct            12606
Corporate          5295
Complementary       743
Aviation            237
Undefined             2
Name: count, dtype: int64

In [15]:
log_data[['distribution_channel']].value_counts()

distribution_channel
TA/TO                   97870
Direct                  14645
Corporate                6677
GDS                       193
Undefined                   5
Name: count, dtype: int64

# encode columns

In [16]:
from sklearn.preprocessing import LabelEncoder

## Encode hotel

Change to one hot encoding

In [17]:
le = LabelEncoder()
log_data['hotel_encoded'] = le.fit_transform(log_data['hotel'])

In [18]:
log_data[['hotel','hotel_encoded']].value_counts()

hotel         hotel_encoded
City Hotel    0                79330
Resort Hotel  1                40060
Name: count, dtype: int64

In [19]:
log_data['is_canceled'].corr(log_data['hotel_encoded'])

-0.13653126949161734

## encode country

In [20]:
le = LabelEncoder()
log_data['country_encoded'] = le.fit_transform(log_data['country'])

In [21]:
log_data[['country','country_encoded','is_canceled']]

,country,country_encoded,is_canceled
0,PRT,135,0
1,PRT,135,0
2,GBR,59,0
3,GBR,59,0
4,GBR,59,0
...,...,...,...
119385,BEL,15,0
119386,FRA,56,0
119387,DEU,43,0
119388,GBR,59,0


In [22]:
log_data['is_canceled'].corr(log_data['country_encoded'])

0.26422339698165603

## encode arrival date month

In [55]:
le = LabelEncoder()
log_data['arrival_date_month_encode'] = le.fit_transform(log_data['arrival_date_month'])

## Room Type

In [40]:
# Check how often the reserved and assigned room types differ
log_data['room_type_mismatch'] = (log_data['reserved_room_type'] != log_data['assigned_room_type']).astype(int)
mismatch_rate = log_data['room_type_mismatch'].mean()
print(f"Mismatch rate: {mismatch_rate:.2%}")


Mismatch rate: 12.49%


In [ ]:
# Calculate cancellation rates for mismatches
mismatch_cancellation_rate = log_data.groupby('room_type_mismatch')['is_canceled'].mean()
print(mismatch_cancellation_rate)

room_type_mismatch
False    0.415629
True     0.053764
Name: is_canceled, dtype: float64


In [36]:
reserved_cancellation_rate = log_data.groupby('reserved_room_type')['is_canceled'].mean()
assigned_cancellation_rate = log_data.groupby('assigned_room_type')['is_canceled'].mean()
print("Reserved Room Type Cancellation Rates:")
print(reserved_cancellation_rate)
print("Assigned Room Type Cancellation Rates:")
print(assigned_cancellation_rate)


Reserved Room Type Cancellation Rates:
reserved_room_type
A    0.391074
B    0.329159
C    0.330472
D    0.317796
E    0.292884
F    0.303763
G    0.364374
H    0.407654
L    0.333333
P    1.000000
Name: is_canceled, dtype: float64
Assigned Room Type Cancellation Rates:
assigned_room_type
A    0.444925
B    0.236708
C    0.187789
D    0.251244
E    0.252114
F    0.247134
G    0.305523
H    0.352528
I    0.013774
K    0.043011
L    1.000000
P    1.000000
Name: is_canceled, dtype: float64


In [41]:
log_data['room_type_mismatch'].value_counts()

room_type_mismatch
0    104473
1     14917
Name: count, dtype: int64

In [42]:
le = LabelEncoder()
log_data['reserved_room_type_encoded'] = le.fit_transform(log_data['reserved_room_type'])
log_data['assigned_room_type_encoded'] = le.fit_transform(log_data['assigned_room_type'])

## Encode deposit type

In [24]:
le = LabelEncoder()
log_data['deposit_encoded'] = le.fit_transform(log_data['deposit_type'])

In [25]:
log_data[['deposit_type','deposit_encoded']].value_counts()

deposit_type  deposit_encoded
No Deposit    0                  104641
Non Refund    1                   14587
Refundable    2                     162
Name: count, dtype: int64

In [26]:
log_data['is_canceled'].corr(log_data['deposit_encoded'])

0.4686338237088315

## Encode Meal 

In [27]:
le = LabelEncoder()
log_data['meal_encoded'] = le.fit_transform(log_data['meal'])

In [28]:
log_data[['meal','meal_encoded']].value_counts()

meal       meal_encoded
BB         0               92310
HB         2               14463
SC         3               10650
Undefined  4                1169
FB         1                 798
Name: count, dtype: int64

In [39]:
log_data['is_canceled'].corr(log_data['room_type_mismatch'])

-0.2477703504391876

In [37]:
log_data.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'babies', 'country', 'market_segment',
       'distribution_channel', 'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type',
       'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'hotel_encoded', 'country_encoded', 'deposit_encoded',
       'room_type_mismatch'],
      dtype='object')

## Encode distribution channel

In [57]:
# drop_first=True argument avoids multicollinearity by dropping the first category 
# and treating it as a reference category.
log_data = pd.get_dummies(log_data, columns=['distribution_channel'], drop_first=True)

In [58]:
log_data.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'country', 'market_segment',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type',
       'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'hotel_encoded', 'country_encoded', 'deposit_encoded',
       'room_type_mismatch', 'reserved_room_type_encoded',
       'assigned_room_type_encoded', 'customer_type_encoded',
       'arrival_date_month_encode', 'distribution_channel_Direct',
       'distribution_channel_GDS', 'distribution_channel_TA/TO',
       'distribution_channel_Undefined'],
      dtype='object')

## Encode customer Type

In [50]:
log_data['customer_type'].value_counts()

customer_type
Transient          89613
Transient-Party    25124
Contract            4076
Group                577
Name: count, dtype: int64

In [51]:
le = LabelEncoder()
log_data['customer_type_encoded'] = le.fit_transform(log_data['customer_type'])

In [53]:
log_data['is_canceled'].corr(log_data['customer_type_encoded'])

-0.06814010513060874

# Checking Correlations

## Remove Meal - very low correlation

In [30]:
log_data['is_canceled'].corr(log_data['meal_encoded'])

-0.01767760995132309

In [31]:
log_data.drop(columns = ['meal','meal_encoded'],inplace=True)

## remove children

In [32]:
log_data['is_canceled'].corr(log_data['children'])

0.005047790029268779

In [33]:
log_data.drop(columns = ['children'],inplace=True)

In [44]:
log_data.columns.dtype

dtype('O')

In [47]:
log_data['is_canceled'].corr(log_data['babies'])

-0.032491089208333185

In [48]:
log_data.drop(columns = ['babies'],inplace=True)

# Remove object columns
keep encoded versions

In [60]:
log_data.drop(columns = ['hotel','arrival_date_month','country','market_segment',
                        'reserved_room_type','assigned_room_type','deposit_type','customer_type'],inplace=True)

In [66]:
log_data.to_csv('log_data_encoded.csv',index=False)